In [1]:
%pip install neurogym

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [22]:
import neurogym as ngym
env = ngym.make('DelayMatchSampleDistractor1D-v0')

In [23]:
import numpy as np

In [24]:
from neurogym.envs.delaymatchsample import DelayMatchSampleDistractor1D


In [25]:
X = []
Y = []
env = DelayMatchSampleDistractor1D()
N = 5
for _ in range(N):
    env.new_trial()
    # input sequence
    x_trial = env.ob # shape: (T, 33)

    # correct labels
    y_trial = env.gt # shape: (T,)
    
    X.append(x_trial)
    Y.append(y_trial)

X = np.stack(X) # shape: (N, T, 33)
Y = np.stack(Y)  # shape: (N, T)


In [26]:
X

array([[[ 1.        ,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        [ 1.        ,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        [ 1.        ,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        ...,
        [ 0.        ,  0.7669887 ,  0.6270695 , ...,  0.9942153 ,
          0.95415807,  0.877433  ],
        [ 0.        ,  0.7669887 ,  0.6270695 , ...,  0.9942153 ,
          0.95415807,  0.877433  ],
        [ 0.        ,  0.7669887 ,  0.6270695 , ...,  0.9942153 ,
          0.95415807,  0.877433  ]],

       [[ 1.        ,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        [ 1.        ,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        [ 1.        ,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        ...,
        [ 0.        , -0.05291424, -0.24671452, ...,  

# RNN

In [27]:
%pip install torch

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [28]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [29]:
"""
FROM DOCS
class DelayMatchSampleDistractor1D(ngym.TrialEnv):
    Delayed match-to-sample with multiple, potentially repeating
    distractors.

    A sample stimulus is shown during the sample period. The stimulus is
    characterized by a one-dimensional variable, such as its orientation
    between 0 and 360 degree. After a delay period, the first test stimulus is
    shown. The agent needs to determine whether the sample and this test
    stimuli are equal. If so, it needs to produce the match response. If the
    first test is not equal to the sample stimulus, another delay period and
    then a second test stimulus follow, and so on.

    metadata = {
        'paper_link': 'https://www.jneurosci.org/content/jneuro/16/16/' +
        '5154.full.pdf',
        'paper_name': '''Neural Mechanisms of Visual Working Memory
        in Prefrontal Cortex of the Macaque''',
        'tags': ['perceptual', 'working memory', 'two-alternative',
                 'supervised']
    }

    def __init__(self, dt=100, rewards=None, timing=None, sigma=1.0):
        super().__init__(dt=dt)
        self.choices = [1, 2, 3]
        self.sigma = sigma / np.sqrt(self.dt)  # Input noise

        # Rewards
        self.rewards = {'abort': -0.1, 'correct': +1., 'fail': -1.}
        if rewards:
            self.rewards.update(rewards)

        self.timing = {
            'fixation': 300,
            'sample': 500,
            'delay1': 1000,
            'test1': 500,
            'delay2': 1000,
            'test2': 500,
            'delay3': 1000,
            'test3': 500}
        if timing:
            self.timing.update(timing)

        self.abort = False

        self.theta = np.arange(0, 2 * np.pi, 2 * np.pi / 32)
#!!!!!! ^^ initializes the preferred angles for each neuron - each neuron's preferred angle is 2pi/32 radians apart (evenly distributed around unit circle) !!!!!

        name = {'fixation': 0, 'stimulus': range(1, 33)}
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(33,),
                                            dtype=np.float32, name=name)

        name = {'fixation': 0, 'match': 1}
        self.action_space = spaces.Discrete(2, name=name)

    def _new_trial(self, **kwargs):
        trial = {
            # There is always a match, ground_truth is which test is a match
            'ground_truth': self.rng.choice(self.choices),
            'sample': self.rng.uniform(0, 2*np.pi),
#!!!!!! ^^ initializes the sample stimulus to a random angle between 0 and 2pi !!!!!
        }
        trial.update(kwargs)

        ground_truth = trial['ground_truth']
        sample = trial['sample']
        for i in [1, 2, 3]:
            tmp = sample if i == ground_truth else self.rng.uniform(0, 2*np.pi)
            trial['test'+str(i)] = tmp
#!!!!!!!!!!!^^^ forms the false "distractor" stimuli for the tests during each working memory task!!!!!!!!!

        periods = ['fixation', 'sample', 'delay1', 'test1',
                   'delay2', 'test2', 'delay3', 'test3']
        self.add_period(periods)

        self.add_ob(1, 'fixation', where='fixation')
        for period in ['sample', 'test1', 'test2', 'test3']:
            self.add_ob(np.cos(self.theta - trial[period]), period, 'stimulus')
#!!!!!!!! ^^ this is the part that creates the stimulus vector based on cos(φ_i-θ)!!!!!!
        self.set_groundtruth(1, 'test'+str(ground_truth))

        return trial

    def _step(self, action):
        new_trial = False
        reward = 0

        ob = self.ob_now
        gt = self.gt_now
        if ((self.in_period('fixation') or self.in_period('sample')) and
                action != 0):
            reward = self.rewards['abort']
            new_trial = self.abort
        elif not self.in_period('test'+str(self.trial['ground_truth'])):
            if action != 0:
                reward = self.rewards['fail']
                new_trial = True
        else:
            if action == 1:
                reward = self.rewards['correct']
                new_trial = True
                self.performance = 1

        return ob, reward, False, {'new_trial': new_trial, 'gt': gt}

"""

'\nFROM DOCS\nclass DelayMatchSampleDistractor1D(ngym.TrialEnv):\n    Delayed match-to-sample with multiple, potentially repeating\n    distractors.\n\n    A sample stimulus is shown during the sample period. The stimulus is\n    characterized by a one-dimensional variable, such as its orientation\n    between 0 and 360 degree. After a delay period, the first test stimulus is\n    shown. The agent needs to determine whether the sample and this test\n    stimuli are equal. If so, it needs to produce the match response. If the\n    first test is not equal to the sample stimulus, another delay period and\n    then a second test stimulus follow, and so on.\n\n    metadata = {\n        \'paper_link\': \'https://www.jneurosci.org/content/jneuro/16/16/\' +\n        \'5154.full.pdf\',\n        \'paper_name\': \'\'\'Neural Mechanisms of Visual Working Memory\n        in Prefrontal Cortex of the Macaque\'\'\',\n        \'tags\': [\'perceptual\', \'working memory\', \'two-alternative\',\n        

In [30]:
def generate_trials(num_trials=1000, distractor_strength=1.0, seed=42):
    """Generate trials for the memory task using NeuroGym."""

    seed = 42 # 67 
    rng = np.random.RandomState(seed)
    env = DelayMatchSampleDistractor1D(dt=20) # dt determines how long each delay / test period is (the test is like fixation + sample + delay1 + test1 + delay2 + test2 + delay3 + test3)
    # also, the sample here is Unif(0, 2pi) and is the angle of the stimulus
    # fixation tells the network to stay still and not respond yet

    all_inputs = []
    all_labels = []

    for _ in range(num_trials):
        env.new_trial()
        # reminder of format:
        # env.ob: network input at each timestep, shape (T, 33)
        # column 0 = fixation signal, columns 1-32 = population-coded stimulus
        # env.gt: correct label at each timestep, shape (T,)
        # 0 = do nothing, 1 = match
        all_inputs.append(env.ob.copy())
        all_labels.append(env.gt.copy())

    X = np.stack(all_inputs) # (num_trials, T, 33)
    Y = np.stack(all_labels) # (num_trials, T)
    return X, Y


class TrialDataset(Dataset):
    """Wraps trial data so PyTorch can batch and shuffle it during training."""

    def __init__(self, X, Y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.long)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]


# one forward pass through the RNN performs one working memory matching task
class VanillaRNN(nn.Module):
    """
    Input = 33 -> 64 recurrent neurons -> Decision (2)

    We use a vanilla RNN (not more advanced versions like LSTM) so the recurrent weight matrix W_hh
    directly represents neuron-to-neuron connection strengths, making it easy
    to analyze like a biological wiring diagram. 64 hidden units is standard
    in computational neuroscience it appears and is not super hard to study.
    """

    def __init__(self, input_size=33, hidden_size=64, output_size=2):
        super().__init__()
        self.hidden_size = hidden_size
        # Recurrent layer with two weight matrices:
        # W_ih: input-to-neuron connections (33 x 64)
        # W_hh: neuron-to-neuron connections (64 x 64), our main analysis target
        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=1,
            nonlinearity='tanh',
            batch_first=True,
        )

        # readout layer reads population activity, outputs a decision
        self.readout = nn.Linear(hidden_size, output_size)

    def forward(self, x, h0=None):
        """Run input through the network. Returns predictions and hidden states."""
        hidden_states, _ = self.rnn(x, h0)
        predictions = self.readout(hidden_states)
        return predictions, hidden_states


# Training
def train_model(
    model, train_loader, val_loader=None, num_epochs=50, learning_rate=1e-3,
    checkpoint_dir='checkpoints', checkpoint_every=10, device='cpu',
    class_weights=None,
):
    """Train the RNN and saves checkpoints"""

    os.makedirs(checkpoint_dir, exist_ok=True)
    model = model.to(device)

    # Cross-entropy loss with class weighting -> important so the network can't cheat by always predicting "do nothing" (90% of timesteps)
    loss_function = nn.CrossEntropyLoss(
        weight=class_weights.to(device) if class_weights is not None else None
    )

    # we use default Adam optimizer for now
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    history = {
        'train_loss': [], 'train_acc': [], 'train_dec_acc': [],
        'val_loss': [], 'val_acc': [], 'val_dec_acc': [],
    }

    for epoch in range(1, num_epochs + 1):

        # Train on batches
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        dec_correct = 0
        dec_total = 0

        for inputs_batch, labels_batch in train_loader:
            inputs_batch = inputs_batch.to(device)
            labels_batch = labels_batch.to(device)

            # forward pass
            predictions, _ = model(inputs_batch)

            loss = loss_function(
                predictions.reshape(-1, predictions.size(-1)),
                labels_batch.reshape(-1),
            )

            # backward pass
            optimizer.zero_grad()
            loss.backward()

            # Clip gradients to prevent exploding gradients (common in vanilla RNNs)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()

            # Track metrics
            running_loss += loss.item() * inputs_batch.size(0)
            predicted_classes = predictions.argmax(dim=-1)
            correct += (predicted_classes == labels_batch).sum().item()
            total += labels_batch.numel()

            #  decision accuracy 
            is_decision = (labels_batch == 1)
            if is_decision.any():
                dec_correct += (predicted_classes[is_decision] == 1).sum().item()
                dec_total += is_decision.sum().item()

        # Record epoch metrics
        train_loss = running_loss / len(train_loader.dataset)
        train_acc = correct / total
        train_dec_acc = dec_correct / dec_total if dec_total > 0 else 0.0

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['train_dec_acc'].append(train_dec_acc)

        # Validate
        val_loss, val_acc, val_dec_acc = None, None, None
        if val_loader is not None:
            val_loss, val_acc, val_dec_acc = evaluate(
                model, val_loader, loss_function, device
            )
            history['val_loss'].append(val_loss)
            history['val_acc'].append(val_acc)
            history['val_dec_acc'].append(val_dec_acc)

        # Print progress
        msg = f"Epoch {epoch:3d}/{num_epochs} | "
        msg += f"Loss: {train_loss:.4f}  DecAcc: {train_dec_acc:.4f}"
        if val_loss is not None:
            msg += f" | ValLoss: {val_loss:.4f}  ValDecAcc: {val_dec_acc:.4f}"
        print(msg)

        # Save checkpoint with recurrent weight matrix for analysis
        if epoch % checkpoint_every == 0 or epoch == num_epochs:
            checkpoint = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'train_loss': train_loss,
                'train_acc': train_acc,
                'val_loss': val_loss,
                'val_acc': val_acc,
                'W_hh': model.rnn.weight_hh_l0.detach().cpu().numpy(),
                'W_ih': model.rnn.weight_ih_l0.detach().cpu().numpy(),
            }
            path = os.path.join(checkpoint_dir, f'epoch_{epoch:03d}.pt')
            torch.save(checkpoint, path)
            print(f"  -> Checkpoint saved: {path}")

    return history


def evaluate(model, loader, loss_function, device='cpu'):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_count = 0
    dec_correct = 0
    dec_count = 0

    with torch.no_grad():
        for inputs_batch, labels_batch in loader:
            inputs_batch = inputs_batch.to(device)
            labels_batch = labels_batch.to(device)

            predictions, _ = model(inputs_batch)
            loss = loss_function(
                predictions.reshape(-1, predictions.size(-1)),
                labels_batch.reshape(-1),
            )

            total_loss += loss.item() * inputs_batch.size(0)
            predicted_classes = predictions.argmax(dim=-1)
            total_correct += (predicted_classes == labels_batch).sum().item()
            total_count += labels_batch.numel()

            is_decision = (labels_batch == 1)
            if is_decision.any():
                dec_correct += (predicted_classes[is_decision] == 1).sum().item()
                dec_count += is_decision.sum().item()

    avg_loss = total_loss / len(loader.dataset)
    accuracy = total_correct / total_count
    dec_accuracy = dec_correct / dec_count if dec_count > 0 else 0.0
    return avg_loss, accuracy, dec_accuracy


# Hidden state extraction for analysis

def extract_hidden_states(model, X, device='cpu'):
    """Run the trained model and record all 64 neurons' activity over time."""
    model.eval()
    X_tensor = torch.tensor(X, dtype=torch.float32).to(device)
    with torch.no_grad():
        _, hidden_states = model(X_tensor)
    return hidden_states.cpu().numpy()




In [31]:
import os

In [32]:
# Main


# Settings for hyperparameters
NUM_TRAIN_TRIALS = 2000
NUM_VAL_TRIALS = 400
BATCH_SIZE = 64
NUM_EPOCHS = 100
LEARNING_RATE = 5e-4
HIDDEN_SIZE = 64
CHECKPOINT_EVERY = 10
DISTRACTOR_STRENGTH = 1.0

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

# Generate data using NeuroGym
print("Generating training trials...")
X_train, Y_train = generate_trials(
    num_trials=NUM_TRAIN_TRIALS,
    distractor_strength=DISTRACTOR_STRENGTH,
    seed=42,
)
print(f"Input shape:  {X_train.shape}")
print(f"Labels shape: {Y_train.shape}")
print(f"Unique labels: {np.unique(Y_train)}")

print("Generating validation trials...")
X_val, Y_val = generate_trials(
    num_trials=NUM_VAL_TRIALS,
    distractor_strength=DISTRACTOR_STRENGTH,
    seed=99,
)

# Compute class weights to handle imbalance
# (most timesteps are "do nothing", only ~10% are actual decisions)
num_classes = len(np.unique(Y_train))
class_counts = np.bincount(Y_train.flatten(), minlength=num_classes)
total_samples = class_counts.sum()
class_weights = torch.tensor(
    total_samples / (num_classes * class_counts),
    dtype=torch.float32,
)
print(f"  Class counts: {dict(enumerate(class_counts))}")
print(f"  Class weights: {class_weights.tolist()}")

# Set up data loaders
train_loader = DataLoader(
    TrialDataset(X_train, Y_train), batch_size=BATCH_SIZE, shuffle=True
)
val_loader = DataLoader(
    TrialDataset(X_val, Y_val), batch_size=BATCH_SIZE, shuffle=False
)

# Create and train the model
model = VanillaRNN(
    input_size=33,
    hidden_size=HIDDEN_SIZE,
    output_size=num_classes,
)
print(f"\nModel architecture:\n{model}")
print(f"Total trainable parameters: {sum(p.numel() for p in model.parameters()):,}")
return 
checkpoint_dir = os.path.join('checkpoints', f'distractor_{DISTRACTOR_STRENGTH}')
history = train_model(
    model, train_loader, val_loader=val_loader,
    num_epochs=NUM_EPOCHS, learning_rate=LEARNING_RATE,
    checkpoint_dir=checkpoint_dir, checkpoint_every=CHECKPOINT_EVERY,
    device=DEVICE, class_weights=class_weights,
)

# Extract hidden states for later analysis (PCA, stability, etc.)
print("\nExtracting hidden states from trained model...")
hidden_states = extract_hidden_states(model, X_val[:50], device=DEVICE)
print(f"Hidden states shape: {hidden_states.shape}")
np.save(os.path.join(checkpoint_dir, 'hidden_states_val.npy'), hidden_states)
print(f"Saved to {checkpoint_dir}/hidden_states_val.npy")

# Save training history for plotting learning curves
np.savez(
    os.path.join(checkpoint_dir, 'training_history.npz'),
    train_loss=history['train_loss'], train_acc=history['train_acc'],
    train_dec_acc=history['train_dec_acc'], val_loss=history['val_loss'],
    val_acc=history['val_acc'], val_dec_acc=history['val_dec_acc'],
)
print("Done! Training history saved.")




Using device: cpu
Generating training trials...
Input shape:  (2000, 265, 33)
Labels shape: (2000, 265)
Unique labels: [0 1]
Generating validation trials...
  Class counts: {0: np.int64(480000), 1: np.int64(50000)}
  Class weights: [0.5520833134651184, 5.300000190734863]

Model architecture:
VanillaRNN(
  (rnn): RNN(33, 64, batch_first=True)
  (readout): Linear(in_features=64, out_features=2, bias=True)
)
Total trainable parameters: 6,466


SyntaxError: 'return' outside function (170723904.py, line 63)

In [33]:
X_train[0]
"""
data format of X_train[0] (the first input of train data):
rows 0–14:   [1, 0, 0, ..., 0]                  ← fixation (repeated)
rows 15–39:  [0, stim1[0], ..., stim1[31]]      ← sample stimulus (same vector repeated)
rows 40–89:  [0, 0, 0, ..., 0]                  ← delay (repeated)
rows 90–114: [0, test1[0], ..., test1[31]]      ← test1 (repeated); could be the "distractor" or the actual stimulus
rows 115–... more delay/test blocks
where row t is a time step in the working memory task; 
stimuli/tests are 33-d vectors since each (apart from v_0 which is the channel that denotes fixation/not fixating)
v_1...v_32 is v_i=cos(φ_i-θ) where φ_i is the preferred angle for neuron i, and θ is the angle in
question that the stimulus is testing (either the reference stimulus or a test stimulus)

data format of Y_train[0] (the first label of train data):
rows 0–14:   0                  ← returns 0 for these time steps bc holding during fixation
rows 15–39:  0     ← returns 0 for these time steps bc holding during sample stimulus
rows 40–89:  0                  ← returns 0 for these time steps bc holding duringdelay
rows 90–114: 1 or 0      ← based on whether the test1 stimulus matches the sample stimulus or the distractor
rows 115–... more responses to delay/test blocks
"""

'\ndata format of X_train[0] (the first input of train data):\nrows 0–14:   [1, 0, 0, ..., 0]                  ← fixation (repeated)\nrows 15–39:  [0, stim1[0], ..., stim1[31]]      ← sample stimulus (same vector repeated)\nrows 40–89:  [0, 0, 0, ..., 0]                  ← delay (repeated)\nrows 90–114: [0, test1[0], ..., test1[31]]      ← test1 (repeated); could be the "distractor" or the actual stimulus\nrows 115–... more delay/test blocks\nwhere row t is a time step in the working memory task; \nstimuli/tests are 33-d vectors since each (apart from v_0 which is the channel that denotes fixation/not fixating)\nv_1...v_32 is v_i=cos(φ_i-θ) where φ_i is the preferred angle for neuron i, and θ is the angle in\nquestion that the stimulus is testing (either the reference stimulus or a test stimulus)\n\ndata format of Y_train[0] (the first label of train data):\nrows 0–14:   0                  ← returns 0 for these time steps bc holding during fixation\nrows 15–39:  0     ← returns 0 for t

In [34]:
Y_train[0][0]

np.int64(0)

In [35]:
print(X.shape)

(5, 53, 33)


In [37]:
import numpy as np

def apply_distractor_scaling(X, Y, alpha):
    """
    X: (N, T, 33)
    Y: (N, T)
    alpha: scalar

    Returns:
        X2 with distractor stimulus scaled by alpha
    """
    X2 = X.copy()

    # stimulus present when any of the 32 channels are nonzero
    stimulus_present = np.linalg.norm(X[:, :, 1:], axis=2) > 0

    # distractor = stimulus present AND not match (Y == 0)
    distractor_mask = (stimulus_present) & (Y == 0)

    # apply scaling only to stimulus channels (1:33)
    X2[distractor_mask, 1:] *= alpha

    return X2

Xdistractor = apply_distractor_scaling(X, Y, 2)

In [41]:
import numpy as np


print(Y)

[[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]]


In [42]:
import numpy as np
import torch

def paper_fidelity_from_input(model, decoder, X_input, theta_true, device='cpu'):
    """
    Compute paper-style fidelity from an input set (e.g. X or X2).

    Parameters
    ----------
    model : trained RNN
        Should return (predictions, hidden_states) when called on X_input.
    decoder : callable
        Maps hidden states -> reconstructed channel responses over angles.
        Expected input/output:
            input:  (N*T, H)
            output: (N*T, K)
        where K is the number of angle bins / channels in the reconstruction.
    X_input : np.ndarray
        Shape (N, T, 33)
    theta_true : np.ndarray
        Shape (N,)
        True sample angle for each trial, in radians.
    device : str
        'cpu' or 'cuda'

    Returns
    -------
    fidelity_t : np.ndarray
        Shape (T,)
        Mean fidelity across trials at each time step.
    recon_aligned : np.ndarray
        Shape (N, T, K)
        Angle-aligned reconstructed channel responses.
    theta_grid_centered : np.ndarray
        Shape (K,)
        Angle grid after centering true angle at 0.
    """

    model.eval()

    # 1) Run input through RNN and get hidden states
    X_tensor = torch.tensor(X_input, dtype=torch.float32, device=device)
    with torch.no_grad():
        _, hidden_states = model(X_tensor)   # (N, T, H)

    hidden_states = hidden_states.detach().cpu().numpy()
    N, T, H = hidden_states.shape

    # 2) Decode / reconstruct channel responses from hidden states
    hidden_flat = hidden_states.reshape(N * T, H)

    with torch.no_grad():
        hidden_flat_torch = torch.tensor(hidden_flat, dtype=torch.float32, device=device)
        recon_flat = decoder(hidden_flat_torch).detach().cpu().numpy()  # (N*T, K)

    recon = recon_flat.reshape(N, T, -1)  # (N, T, K)
    K = recon.shape[-1]

    # 3) Build angle grid for the reconstruction channels
    theta_grid = np.linspace(0, 2 * np.pi, K, endpoint=False)

    # 4) Align each trial so the true sample angle is at 0
    #    This matches the paper logic: align reconstruction to the true remembered angle.
    recon_aligned = np.zeros_like(recon)

    for n in range(N):
        # shift amount in bins
        shift_bins = int(np.round((theta_true[n] / (2 * np.pi)) * K)) % K

        # roll so theta_true is centered at 0
        recon_aligned[n] = np.roll(recon[n], -shift_bins, axis=-1)

    # centered grid after alignment
    theta_grid_centered = np.linspace(0, 2 * np.pi, K, endpoint=False)

    # 5) Paper-style fidelity:
    #    F(t) = mean_theta( r(theta) * cos(theta) )
    # after alignment, true angle is at 0
    cos_weights = np.cos(theta_grid_centered)  # (K,)

    fidelity_trials_t = np.mean(recon_aligned * cos_weights[None, None, :], axis=-1)  # (N, T)
    fidelity_t = fidelity_trials_t.mean(axis=0)  # (T,)

    return fidelity_t, recon_aligned, theta_grid_centered

In [ ]:
import matplotlib.pyplot as plt
fidelity_X, _, _ = paper_fidelity_from_input(model, decoder, X, theta_true, device='cpu')
fidelity_X2, _, _ = paper_fidelity_from_input(model, decoder, X2, theta_true, device='cpu')

plt.plot(fidelity_X, label='X (baseline)')
plt.plot(fidelity_X2, label='X2 (with scaled distractor)')
plt.xlabel('Time step')
plt.ylabel('Fidelity')
plt.legend()
plt.show()